# Notebook 0 — Laplace approximation in 1-D

**Concetto (dispense §7.3).** Vogliamo approssimare una densità che conosciamo solo
a meno della costante di normalizzazione,
$$p(z) = \frac{1}{Z} f(z), \qquad Z = \int f(z)\,dz ,$$
con una **Gaussiana**. L'idea è locale in due passi:

1. **modo** — trova $z_0$ tale che $\dfrac{d}{dz}\log f(z_0)=0$ (un massimo);
2. **curvatura** — sviluppa $\log f$ al second'ordine attorno a $z_0$:
$$\log f(z)\approx \log f(z_0) - \tfrac12 A\,(z-z_0)^2,\qquad A=-\frac{d^2}{dz^2}\log f(z_0)>0 .$$

Esponenziando si ottiene $f(z)\approx f(z_0)\exp\!\big(-\tfrac12 A(z-z_0)^2\big)$, cioè
$$\boxed{\,q(z)=\mathcal N\!\big(z\mid z_0,\;A^{-1}\big)\,}$$
e come sottoprodotto una stima della costante di normalizzazione
$$Z \approx f(z_0)\sqrt{\tfrac{2\pi}{A}} .$$

**Attenzione (limite del metodo).** È un'approssimazione *locale*: cattura solo modo e
curvatura, non la forma globale. Se $p$ è asimmetrica o ha più modi, la Gaussiana sbaglia.

**Prova di comprensione (obiettivo del notebook).** Applicare il metodo a una densità
nota, ricavare $q(z)=\mathcal N(z_0,A^{-1})$ *a mano*, e confrontare la stima di $Z$
con l'integrale vero calcolato per quadratura.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # evita OMP Error #15 (Windows)

import sys, pathlib
# root del progetto = cartella che contiene 'src', trovata a partire dal notebook
here = pathlib.Path.cwd()
for base in [here, *here.parents]:
    if (base / "code_v2" / "src" / "laplace_core.py").is_file():
        sys.path.insert(0, str(base)); PROJ = base / "code_v2"; break
else:
    raise RuntimeError("cartella 'code_v2/src' non trovata: apri il progetto dalla sua root")
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

from code_v2.src.laplace_core import fit_laplace_1d, gaussian_pdf

np.set_printoptions(precision=6, suppress=True)

## La densità di prova

Scegliamo una densità **asimmetrica** e con supporto $z>0$, così la Laplace è davvero
un'approssimazione (non esatta come sarebbe su una Gaussiana), ma con $Z$ noto in forma chiusa:
$$f(z) = z^2 e^{-z},\quad z>0 .$$
È una $\text{Gamma}(3,1)$ non normalizzata, quindi il vero normalizzatore è
$Z=\int_0^\infty z^2 e^{-z}\,dz=\Gamma(3)=2!=2$.

In [ ]:
f     = lambda z: z**2 * np.exp(-z)      # densità NON normalizzata
log_f = lambda z: 2*np.log(z) - z         # il suo logaritmo (z > 0)

## Derivazione a mano (poi la confrontiamo col codice)

$\log f(z)=2\log z - z$.

- **Modo:** $\dfrac{d}{dz}\log f = \dfrac{2}{z}-1 = 0 \;\Rightarrow\; z_0 = 2$.
- **Curvatura:** $\dfrac{d^2}{dz^2}\log f = -\dfrac{2}{z^2}$, quindi
  $A = -\left(-\dfrac{2}{z_0^2}\right)=\dfrac{2}{4}=0.5$ e varianza $A^{-1}=2$.
- **Normalizzatore:** $Z\approx f(2)\sqrt{2\pi/A}=4e^{-2}\sqrt{4\pi}\approx 1.919$.

In [ ]:
lap = fit_laplace_1d(log_f, z_init=1.0, bounds=(1e-6, 50))

print(f"modo z0      : {lap.mode:.6f}   (a mano: 2)")
print(f"precisione A : {lap.precision:.6f}   (a mano: 0.5)")
print(f"varianza 1/A : {lap.variance:.6f}   (a mano: 2)")

## Confronto con il vero $Z$ (quadratura numerica)

In [ ]:
logZ_lap = lap.log_normalizer(np.log(f(lap.mode)))
Z_lap = np.exp(logZ_lap)
Z_true, _ = quad(f, 0, np.inf)     # = Gamma(3) = 2

print(f"Z  Laplace     : {Z_lap:.6f}")
print(f"Z  vero (quad) : {Z_true:.6f}")
print(f"errore rel.    : {abs(Z_lap - Z_true)/Z_true*100:.2f}%")

Laplace **sottostima** $Z$ di circa il 4%: è la firma dell'asimmetria. La Gaussiana,
simmetrica, non può riprodurre la coda destra più lunga della Gamma.

## Visualizzazione: densità vera vs Gaussiana di Laplace

In [ ]:
z = np.linspace(1e-3, 10, 400)
p_true = f(z) / Z_true          # densità vera normalizzata
q = lap.pdf(z)                  # q(z) = N(z0, A^-1)

plt.figure(figsize=(7,4))
plt.plot(z, p_true, label="densità vera  p(z)=f(z)/Z", lw=2)
plt.plot(z, q, "--", label=r"Laplace  q(z)=$\mathcal{N}(z_0, A^{-1})$", lw=2)
plt.axvline(lap.mode, color="k", ls=":", alpha=.6, label=f"modo z0={lap.mode:.2f}")
plt.xlabel("z"); plt.ylabel("densità"); plt.legend(); plt.title("Laplace 1-D su una Gamma(3,1)")
plt.tight_layout(); plt.show()

## Quando NON regge

Il grafico mostra i due difetti tipici di una Laplace su densità non gaussiana:
la Gaussiana **mette massa in $z<0$** (dove la vera densità è nulla) e **non cattura
l'asimmetria** della coda. Sono esattamente i casi che nel Notebook 2 ci faranno
distinguere *quando* la last-layer Laplace è affidabile e quando no.

**Riassunto.** Modo + curvatura $\to$ $q(z)=\mathcal N(z_0,A^{-1})$; la stessa idea, in
$n$ dimensioni con $A=-\nabla\nabla\log f(z_0)$, è ciò che useremo sui pesi dell'ultimo
layer nei notebook successivi.